In [94]:
import numpy as np
import pandas as pd

In [95]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')

In [96]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Gerpany?,Berlin
2,Who wrote 'To Kill a sockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [97]:
#tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?','')
    text = text.replace("'","")
    return text.split()

In [98]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [99]:
#vocab
vocab = {'<UNK>':0}


In [100]:
def build_vocab(row):
    
    tokenized_question = tokenize(row['question'])
    tokenized_answer   = tokenize(row['answer'])
    
    merge_token = tokenized_question +tokenized_answer
    
    for token in merge_token:
        if token not in vocab:
            vocab[token] = len(vocab)
    

In [101]:
df.apply(build_vocab , axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [102]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'gerpany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'sockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [103]:
#convert words to numerical indices
def text_to_indices(text, vocab):
    indexed_text = []
    
    for token in tokenize(text):
        
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
            
    return indexed_text

In [104]:
text_to_indices("what is campusx", vocab)

[1, 2, 0]

In [105]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'gerpany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'sockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [106]:
import torch
from torch.utils.data import Dataset , DataLoader

In [107]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df 
        self.vocab = vocab

    def __getitem__(self, index):
        
        numerical_question = text_to_indices(self.df.iloc[index]['question'] , self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'] , self.vocab)
        
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)

    def __len__(self):
        return self.df.shape[0]

In [108]:
dataset = QADataset(df , vocab)

In [109]:
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 53]), tensor([54]))

In [110]:
dataloader = DataLoader(dataset , batch_size=1 , shuffle=True)

In [111]:
for q ,a in dataloader:
    print(q, a)

tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]]) tensor([[268]])
tensor([[ 42, 137, 118,   3, 247,   5, 248]]) tensor([[249]])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([[85]])
tensor([[  1,   2,   3,   4,   5, 279]]) tensor([[280]])
tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([[215]])
tensor([[ 42, 101,   2,   3,  17]]) tensor([[102]])
tensor([[ 42,  86,  87, 241, 242,  19,  39, 243]]) tensor([[244]])
tensor([[ 42, 200,   2,  14, 201, 202, 203, 204]]) tensor([[205]])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([[156]])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([[205]])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([[36]])
tensor([[ 10,  75, 208]]) tensor([[209]])
tensor([[  1,   2,   3,   4,   5, 206]]) tensor([[207]])
tensor([[ 10,  75, 111]]) tensor([[112]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor(

In [112]:
import torch.nn as nn

In [113]:
class SimpleRNN(nn.Module):
    def __init__(self , vocab_size):
        
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size , embedding_dim=50)
        self.rnn= nn.RNN(50, 64 , batch_first=True)
        self.fc = nn.Linear(64, vocab_size)
        

    def forward(self, question):
        embedded_q = self.embedding(question)
        hidden , final = self.rnn(embedded_q)
        output = self.fc(final.squeeze(0))
        
        return output
        

In [114]:
x= nn.Embedding(324 , embedding_dim=50)

In [115]:
dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [116]:
x(dataset[0][0]).shape

torch.Size([6, 50])

In [117]:
x(dataset[10][0]).shape

torch.Size([6, 50])

In [118]:
dataset[10][0]

tensor([ 1,  2,  3,  4,  5, 53])

In [119]:
a = x(dataset[10][0])

In [120]:
y = nn.RNN(50, 64)

In [121]:
y(a)[1].shape

torch.Size([1, 64])

In [122]:
b = y(a)[1]# give two output but sequential need one thats why sequential dont work we need manual coding

In [123]:
z = nn.Linear(64, 324)

In [124]:
z(b).shape

torch.Size([1, 324])

In [125]:
learning_rate = 0.001
epochs=100

In [126]:
model = SimpleRNN(len(vocab))

In [127]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr= learning_rate )

In [128]:

for epoch in range(epochs):
    total_loss =0
    for question , answer in dataloader:
        # move data to gpu
        # batch_features = batch_features.to(device)
        # batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        #forward pass
        output = model(question)

        #loss calculate
        loss = criterion(output, answer[0])

        #zero grad for next epoch not needed accumulate for grad
        # It is better to make it zero before calculation than after
       
        #backward pass
        loss.backward()
        #parameter update
        optimizer.step()
        total_loss = total_loss + loss.item()

    print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 526.747428
Epoch: 2, Loss: 458.785894
Epoch: 3, Loss: 377.501583
Epoch: 4, Loss: 317.456542
Epoch: 5, Loss: 266.338754
Epoch: 6, Loss: 218.636350
Epoch: 7, Loss: 173.959533
Epoch: 8, Loss: 135.184509
Epoch: 9, Loss: 103.876920
Epoch: 10, Loss: 79.227562
Epoch: 11, Loss: 61.780957
Epoch: 12, Loss: 47.331127
Epoch: 13, Loss: 37.637983
Epoch: 14, Loss: 30.226037
Epoch: 15, Loss: 24.899858
Epoch: 16, Loss: 20.761802
Epoch: 17, Loss: 17.444980
Epoch: 18, Loss: 14.783893
Epoch: 19, Loss: 12.805719
Epoch: 20, Loss: 11.090304
Epoch: 21, Loss: 9.597592
Epoch: 22, Loss: 8.533052
Epoch: 23, Loss: 7.565693
Epoch: 24, Loss: 6.741591
Epoch: 25, Loss: 6.030111
Epoch: 26, Loss: 5.438491
Epoch: 27, Loss: 4.932654
Epoch: 28, Loss: 4.475879
Epoch: 29, Loss: 4.084939
Epoch: 30, Loss: 3.748709
Epoch: 31, Loss: 3.443602
Epoch: 32, Loss: 3.168146
Epoch: 33, Loss: 2.921906
Epoch: 34, Loss: 2.712229
Epoch: 35, Loss: 2.512805
Epoch: 36, Loss: 2.330257
Epoch: 37, Loss: 2.170933
Epoch: 38, Loss: 2

In [131]:
def predict(model , question , threshold=0.5):
    
    #conver question to numbers 
    numerical_q = text_to_indices(question, vocab)
    
    # tensor
    question_tensor = torch.tensor(numerical_q).unsqueeze(0)
    
    output = model(question_tensor)
    
    #convert logit to probs
    probs = torch.nn.functional.softmax(output, dim=1)
    
    value , index = torch.max(probs , dim=1)
    
    if value< threshold:
        print("I don't know!")
    else:
        print(list(vocab.keys())[index])
    

In [132]:
predict(model, "Which element has the atomic number 1?")

hydrogen
